# Siamese Network for Resume-Job Description Matching

## 💾 Checkpoint System

This notebook includes **automatic checkpoint saving/loading** to avoid retraining:

- **First Run**: Trains the model and saves checkpoint + encoder files
- **Subsequent Runs**: Automatically loads from checkpoint, skips training
- **Force Retrain**: Set `FORCE_RETRAIN = True` to retrain even with existing checkpoint
- **Management**: Use utility functions to view/delete checkpoints

**Files Created:**
- `../models/siamese_checkpoint.pth` - Complete model checkpoint with training state
- `../models/siamese_encoder/` - Saved encoder for inference

## Phase 1: Data Preparation

In [15]:
import pandas as pd
import os
import random
from tqdm import tqdm

tqdm.pandas()

In [16]:
# Define paths
RESUME_PATH = '../data/raw/parsed_resumes.csv'
JD_FOLDER_PATH = '../data/job_descriptions/'
OUTPUT_PATH = '../data/processed/triplet_training_data.csv'

# Define the target job description for positive examples
TARGET_JD_FILENAME = 'Web-Developer-job-description.txt'

In [17]:
# Load Resumes
resumes_df = pd.read_csv(RESUME_PATH)

# Concatenate relevant fields to create Resume_str
fields_to_concat = [
    'Person Name', 'Work Experience', 'Skills', 'Education', 'Certifications', 'Projects', 'Summary', 'Contact Information'
]

# Only use fields that exist in the dataframe
fields_to_concat = [f for f in fields_to_concat if f in resumes_df.columns]

resumes_df['Resume_str'] = resumes_df[fields_to_concat].fillna('').agg(' '.join, axis=1)

print(f'Loaded {len(resumes_df)} resumes. Created Resume_str by concatenating: {fields_to_concat}')

Loaded 2437 resumes. Created Resume_str by concatenating: ['Person Name', 'Work Experience', 'Skills', 'Education']


In [18]:
# Load Job Descriptions
jd_files = os.listdir(JD_FOLDER_PATH)
job_descriptions = {}
for file_name in jd_files:
    with open(os.path.join(JD_FOLDER_PATH, file_name), 'r', encoding='utf-8') as f:
        job_descriptions[file_name] = f.read()

print(f'Loaded {len(job_descriptions)} job descriptions.')

Loaded 5 job descriptions.


### Generate Triplets
We will create a triplet for each resume. The resume is the **anchor**. The **positive** example is the target job description ('Web-Developer-job-description.txt'). The **negative** example is any other job description chosen at random.

In [19]:
positive_jd = job_descriptions[TARGET_JD_FILENAME]
negative_jd_files = [f for f in jd_files if f != TARGET_JD_FILENAME]

triplets = []
for index, row in tqdm(resumes_df.iterrows(), total=resumes_df.shape[0]):
    # Anchor is the resume text
    anchor = row['Resume_str']
    
    # Positive is the target JD
    positive = positive_jd
    
    # Negative is a randomly chosen different JD
    negative_filename = random.choice(negative_jd_files)
    negative = job_descriptions[negative_filename]
    
    triplets.append({'anchor': anchor, 'positive': positive, 'negative': negative})

triplets_df = pd.DataFrame(triplets)
print(f'Generated {len(triplets_df)} triplets.')
triplets_df.head()

100%|██████████| 2437/2437 [00:00<00:00, 49224.51it/s]

Generated 2437 triplets.


,anchor,positive,negative
0,A senior systems administrator trico products ...,\nJob Title: Web Developer\nCompany: Not speci...,Position: Project Manager\nExperience: 3-6 Yea...
1,B systems administrator bios technologies - me...,\nJob Title: Web Developer\nCompany: Not speci...,Position: Data Scientist\nExperience: 2-4 Year...
2,C systems administrator nord gear corporation ...,\nJob Title: Web Developer\nCompany: Not speci...,Position: Software Engineer\nExperience: 1-3 Y...
3,"D roti mediterranean grill - north bethesda, m...",\nJob Title: Web Developer\nCompany: Not speci...,Position: Data Scientist\nExperience: 2-4 Year...
4,E systems administrator bex realty - boca rato...,\nJob Title: Web Developer\nCompany: Not speci...,Position: Data Scientist\nExperience: 2-4 Year...


In [20]:
# Save the triplets to a new CSV file
triplets_df.to_csv(OUTPUT_PATH, index=False)
print(f'Triplet data saved to {OUTPUT_PATH}')

Triplet data saved to ../data/processed/triplet_training_data.csv


## Phase 2 & 3: Model Architecture & Training

In [21]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
import torch.nn.functional as F

### Configuration

In [22]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 1
LEARNING_RATE = 2e-5

MARGIN = 0.5

In [23]:
# K-Fold Cross-Validation Configuration
from sklearn.model_selection import KFold
import numpy as np

K_FOLDS = 5  # Number of folds for cross-validation
RANDOM_STATE = 42  # For reproducible results

print(f"K-Fold Cross-Validation Configuration:")
print(f"Number of folds: {K_FOLDS}")
print(f"Random state: {RANDOM_STATE}")

K-Fold Cross-Validation Configuration:
Number of folds: 5
Random state: 42


### Create a PyTorch Dataset

In [24]:
class TripletDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        def tokenize(text):
            tokens = self.tokenizer.tokenize(text)
            tokens = tokens[:self.max_length - 2]
            input_ids = self.tokenizer.convert_tokens_to_ids(tokens)
            input_ids = self.tokenizer.build_inputs_with_special_tokens(input_ids)
            attention_mask = [1] * len(input_ids)
            padding_length = self.max_length - len(input_ids)
            input_ids = input_ids + ([self.tokenizer.pad_token_id] * padding_length)
            attention_mask = attention_mask + ([0] * padding_length)
            return torch.tensor(input_ids), torch.tensor(attention_mask)
        anchor_input_ids, anchor_attention_mask = tokenize(row['anchor'])
        positive_input_ids, positive_attention_mask = tokenize(row['positive'])
        negative_input_ids, negative_attention_mask = tokenize(row['negative'])
        return {
            'anchor': {'input_ids': anchor_input_ids, 'attention_mask': anchor_attention_mask},
            'positive': {'input_ids': positive_input_ids, 'attention_mask': positive_attention_mask},
            'negative': {'input_ids': negative_input_ids, 'attention_mask': negative_attention_mask}
        }

### Define the Siamese Network Architecture

In [25]:
class SiameseNetwork(nn.Module):
    def __init__(self, model_name):
        super(SiameseNetwork, self).__init__()
        self.encoder = DistilBertModel.from_pretrained(model_name)

    def forward_once(self, input_ids, attention_mask):
        
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use mean pooling for sentence representation
        pooled_output = outputs[0].mean(axis=1)
        return pooled_output

    def forward(self, anchor, positive, negative):
        anchor_embedding = self.forward_once(anchor['input_ids'], anchor['attention_mask'])
        positive_embedding = self.forward_once(positive['input_ids'], positive['attention_mask'])
        negative_embedding = self.forward_once(negative['input_ids'], negative['attention_mask'])
        return anchor_embedding, positive_embedding, negative_embedding

### Define the Triplet Loss

In [26]:
class TripletLoss(nn.Module):
    def __init__(self, margin):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        distance_positive = F.pairwise_distance(anchor, positive, p=2)
        distance_negative = F.pairwise_distance(anchor, negative, p=2)
        loss = torch.mean(F.relu(distance_positive - distance_negative + self.margin))
        return loss

### Training Setup

In [27]:
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
train_dataset = TripletDataset(triplets_df, tokenizer, MAX_LENGTH)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Dataset and DataLoader prepared')
print(f'Training dataset size: {len(train_dataset)}')
print(f'Number of batches: {len(train_dataloader)}')

Dataset and DataLoader prepared
Training dataset size: 2437
Number of batches: 153


In [28]:
# Model Checkpoint Configuration
CHECKPOINT_PATH = '../models/siamese_checkpoint.pth'
FORCE_RETRAIN = False  # Set to True to force retraining even if checkpoint exists

# Check if trained model checkpoint exists
import os
checkpoint_exists = os.path.exists(CHECKPOINT_PATH)

print(f"Checkpoint Configuration:")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {checkpoint_exists}")
print(f"Force retrain: {FORCE_RETRAIN}")

if checkpoint_exists and not FORCE_RETRAIN:
    print("✅ Found existing checkpoint - will load trained model")
else:
    print("🔄 Will train model from scratch")

Checkpoint Configuration:
Checkpoint path: ../models/siamese_checkpoint.pth
Checkpoint exists: False
Force retrain: False
🔄 Will train model from scratch


In [29]:
# Initialize or Load Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if checkpoint_exists and not FORCE_RETRAIN:
    print("📂 Loading model from checkpoint...")
    
    # Load checkpoint
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    
    # Initialize model
    model = SiameseNetwork(MODEL_NAME).to(device)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer state
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load training info
    start_epoch = checkpoint['epoch']
    best_loss = checkpoint['loss']
    
    print(f"✅ Model loaded successfully!")
    print(f"   - Trained for {start_epoch} epochs")
    print(f"   - Best loss: {best_loss:.4f}")
    print(f"   - Checkpoint created: {checkpoint.get('timestamp', 'Unknown')}")
    
    # Set model to evaluation mode
    model.eval()
    
else:
    print("🔄 Initializing new model for training...")
    
    # Initialize new model
    model = SiameseNetwork(MODEL_NAME).to(device)
    loss_fn = TripletLoss(margin=MARGIN)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    start_epoch = 0
    best_loss = float('inf')
    
    print("✅ New model initialized successfully!")

print(f"Model is ready on {device}")

Using device: cpu
🔄 Initializing new model for training...
✅ New model initialized successfully!
Model is ready on cpu


### Training Loop (Conditional)

In [30]:
# Conditional Training - Skip if model already loaded
if checkpoint_exists and not FORCE_RETRAIN:
    print("⏭️  SKIPPING TRAINING - Model already trained and loaded from checkpoint")
    print(f"   To retrain, set FORCE_RETRAIN = True or delete {CHECKPOINT_PATH}")
    
else:
    print("🚀 STARTING TRAINING...")
    
    # Initialize loss function if not already done
    if 'loss_fn' not in locals():
        loss_fn = TripletLoss(margin=MARGIN)
    
    model.train()
    training_losses = []
    
    for epoch in range(EPOCHS):
        total_loss = 0
        epoch_batches = 0
        
        print(f"\n--- Epoch {epoch + 1}/{EPOCHS} ---")
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
            try:
                optimizer.zero_grad()
                anchor = {k: v.to(device) for k, v in batch['anchor'].items()}
                positive = {k: v.to(device) for k, v in batch['positive'].items()}
                negative = {k: v.to(device) for k, v in batch['negative'].items()}

                anchor_embedding, positive_embedding, negative_embedding = model(anchor, positive, negative)
                loss = loss_fn(anchor_embedding, positive_embedding, negative_embedding)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                epoch_batches += 1
                
            except Exception as e:
                print(f"Error in batch: {e}")
                continue
        
        avg_loss = total_loss / epoch_batches if epoch_batches > 0 else 0
        training_losses.append(avg_loss)
        print(f'Epoch {epoch + 1}/{EPOCHS}, Average Loss: {avg_loss:.4f}')
        
        # Update best loss
        if avg_loss < best_loss:
            best_loss = avg_loss
    
    print(f"\n✅ Training completed!")
    print(f"Final average loss: {training_losses[-1]:.4f}")
    print(f"Best loss achieved: {best_loss:.4f}")

🚀 STARTING TRAINING...

--- Epoch 1/1 ---


Epoch 1/1: 100%|██████████| 153/153 [34:27<00:00, 13.51s/it]

Epoch 1/1, Average Loss: 0.0018

✅ Training completed!
Final average loss: 0.0018
Best loss achieved: 0.0018


In [40]:
# Save Model Checkpoint
if not (checkpoint_exists and not FORCE_RETRAIN):
    print("\n💾 SAVING MODEL CHECKPOINT...")
    
    # Create checkpoint directory if it doesn't exist
    checkpoint_dir = os.path.dirname(CHECKPOINT_PATH)
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Prepare checkpoint data
    from datetime import datetime
    checkpoint_data = {
        'epoch': EPOCHS,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': best_loss,
        'training_losses': training_losses if 'training_losses' in locals() else [],
        'config': {
            'MODEL_NAME': MODEL_NAME,
            'MAX_LENGTH': MAX_LENGTH,
            'BATCH_SIZE': BATCH_SIZE,
            'EPOCHS': EPOCHS,
            'LEARNING_RATE': LEARNING_RATE,
            'MARGIN': MARGIN,
            'K_FOLDS': K_FOLDS
        },
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    # Save checkpoint
    torch.save(checkpoint_data, CHECKPOINT_PATH)
    
    print(f"✅ Checkpoint saved successfully to: {CHECKPOINT_PATH}")
    print(f"   - Model state, optimizer state, and training history saved")
    print(f"   - Timestamp: {checkpoint_data['timestamp']}")
    
else:
    print("⏭️  Checkpoint already exists - skipping save")

# Ensure model is in evaluation mode for inference
model.eval()
print("\n🎯 Model ready for inference!")


💾 SAVING MODEL CHECKPOINT...
✅ Checkpoint saved successfully to: ../models/siamese_checkpoint.pth
   - Model state, optimizer state, and training history saved
   - Timestamp: 2025-07-31 01:32:30

🎯 Model ready for inference!
✅ Checkpoint saved successfully to: ../models/siamese_checkpoint.pth
   - Model state, optimizer state, and training history saved
   - Timestamp: 2025-07-31 01:32:30

🎯 Model ready for inference!


### Save the Model (Conditional)

In [51]:
# Save Model Encoder (Conditional)
ENCODER_SAVE_PATH = '../models/siamese_encoder'

# Check if encoder already exists
encoder_exists = os.path.exists(ENCODER_SAVE_PATH) and os.listdir(ENCODER_SAVE_PATH)

if not encoder_exists or not (checkpoint_exists and not FORCE_RETRAIN):
    print(f"💾 Saving model encoder to: {ENCODER_SAVE_PATH}")
    
    os.makedirs(ENCODER_SAVE_PATH, exist_ok=True)
    
    try:
        # Clear GPU memory and move model to CPU temporarily for saving
        model.encoder.cpu()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        # Save with safe_serialization=False to avoid SafetensorError
        model.encoder.save_pretrained(ENCODER_SAVE_PATH, safe_serialization=False)
        tokenizer.save_pretrained(ENCODER_SAVE_PATH)
        
        # Move model back to original device
        model.encoder.to(device)
        
        print(f'✅ Model encoder saved successfully!')
        print(f'   - Model files: {ENCODER_SAVE_PATH}')
        print(f'   - Tokenizer files: {ENCODER_SAVE_PATH}')
        
    except Exception as e:
        print(f"❌ Error saving encoder: {e}")
        print("   Trying alternative save method...")
        
        # Alternative: Save using torch.save
        import shutil
        if os.path.exists(ENCODER_SAVE_PATH):
            shutil.rmtree(ENCODER_SAVE_PATH)
        os.makedirs(ENCODER_SAVE_PATH, exist_ok=True)
        
        # Save model state dict manually
        torch.save(model.encoder.state_dict(), os.path.join(ENCODER_SAVE_PATH, 'pytorch_model.bin'))
        tokenizer.save_pretrained(ENCODER_SAVE_PATH)
        
        # Save config manually
        model.encoder.config.save_pretrained(ENCODER_SAVE_PATH)
        
        print(f'✅ Model encoder saved using alternative method!')
    
else:
    print(f"⏭️  Encoder already exists at: {ENCODER_SAVE_PATH}")
    print("   Skipping encoder save")

print(f"\n📁 Model files available at:")
print(f"   - Checkpoint: {CHECKPOINT_PATH}")
print(f"   - Encoder: {ENCODER_SAVE_PATH}")

💾 Saving model encoder to: ../models/siamese_encoder
✅ Model encoder saved successfully!
   - Model files: ../models/siamese_encoder
   - Tokenizer files: ../models/siamese_encoder

📁 Model files available at:
   - Checkpoint: ../models/siamese_checkpoint.pth
   - Encoder: ../models/siamese_encoder
✅ Model encoder saved successfully!
   - Model files: ../models/siamese_encoder
   - Tokenizer files: ../models/siamese_encoder

📁 Model files available at:
   - Checkpoint: ../models/siamese_checkpoint.pth
   - Encoder: ../models/siamese_encoder


### Checkpoint Management Utilities

Use these utilities to manage your trained model checkpoints:

In [52]:
# Checkpoint Management Functions

def check_checkpoint_info():
    """Display information about existing checkpoint"""
    if os.path.exists(CHECKPOINT_PATH):
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
        print("📊 CHECKPOINT INFORMATION:")
        print(f"   File: {CHECKPOINT_PATH}")
        print(f"   Timestamp: {checkpoint.get('timestamp', 'Unknown')}")
        print(f"   Epochs trained: {checkpoint.get('epoch', 'Unknown')}")
        print(f"   Final loss: {checkpoint.get('loss', 'Unknown'):.4f}")
        print(f"   Model config: {checkpoint.get('config', {})}")
        
        if 'training_losses' in checkpoint and checkpoint['training_losses']:
            losses = checkpoint['training_losses']
            print(f"   Training losses: {losses}")
            print(f"   Loss improvement: {losses[0]:.4f} → {losses[-1]:.4f}")
        
        file_size = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)  # MB
        print(f"   File size: {file_size:.2f} MB")
    else:
        print("❌ No checkpoint found")

def delete_checkpoint():
    """Delete existing checkpoint to force retraining"""
    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)
        print(f"🗑️  Checkpoint deleted: {CHECKPOINT_PATH}")
        print("   Next run will train from scratch")
    else:
        print("❌ No checkpoint to delete")

def delete_encoder():
    """Delete saved encoder files"""
    import shutil
    if os.path.exists(ENCODER_SAVE_PATH):
        shutil.rmtree(ENCODER_SAVE_PATH)
        print(f"🗑️  Encoder deleted: {ENCODER_SAVE_PATH}")
    else:
        print("❌ No encoder to delete")

def reset_all():
    """Delete both checkpoint and encoder files"""
    delete_checkpoint()
    delete_encoder()
    print("🔄 All model files deleted - fresh start on next run")

# Display current status
print("🔧 CHECKPOINT MANAGEMENT UTILITIES:")
print("   - check_checkpoint_info(): View checkpoint details")
print("   - delete_checkpoint(): Remove checkpoint (force retrain)")
print("   - delete_encoder(): Remove encoder files")
print("   - reset_all(): Remove all model files")
print("\n📊 Current Status:")
check_checkpoint_info()

🔧 CHECKPOINT MANAGEMENT UTILITIES:
   - check_checkpoint_info(): View checkpoint details
   - delete_checkpoint(): Remove checkpoint (force retrain)
   - delete_encoder(): Remove encoder files
   - reset_all(): Remove all model files

📊 Current Status:
📊 CHECKPOINT INFORMATION:
   File: ../models/siamese_checkpoint.pth
   Timestamp: 2025-07-31 01:32:30
   Epochs trained: 1
   Final loss: 0.0018
   Model config: {'MODEL_NAME': 'distilbert-base-uncased', 'MAX_LENGTH': 256, 'BATCH_SIZE': 16, 'EPOCHS': 1, 'LEARNING_RATE': 2e-05, 'MARGIN': 0.5, 'K_FOLDS': 5}
   Training losses: [0.001786950561735365]
   Loss improvement: 0.0018 → 0.0018
   File size: 759.58 MB
📊 CHECKPOINT INFORMATION:
   File: ../models/siamese_checkpoint.pth
   Timestamp: 2025-07-31 01:32:30
   Epochs trained: 1
   Final loss: 0.0018
   Model config: {'MODEL_NAME': 'distilbert-base-uncased', 'MAX_LENGTH': 256, 'BATCH_SIZE': 16, 'EPOCHS': 1, 'LEARNING_RATE': 2e-05, 'MARGIN': 0.5, 'K_FOLDS': 5}
   Training losses: [0.001786

## Phase 4: Inference and Ranking

In [53]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

### Load the Fine-Tuned Encoder

In [54]:
ENCODER_PATH = '../models/siamese_encoder'
RANKING_OUTPUT_PATH = '../data/results/siamese_ranking_results.csv'

tokenizer = DistilBertTokenizer.from_pretrained(ENCODER_PATH)
encoder = DistilBertModel.from_pretrained(ENCODER_PATH)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder.to(device)
encoder.eval()
print('Model loaded and in evaluation mode.')

Model loaded and in evaluation mode.


### Function to Generate Embeddings

In [55]:
def get_embedding(text, tokenizer, model, device, max_length=256):
    tokens = tokenizer.tokenize(text)
    tokens = tokens[:max_length - 2]
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = tokenizer.build_inputs_with_special_tokens(input_ids)
    attention_mask = [1] * len(input_ids)
    padding_length = max_length - len(input_ids)
    input_ids = input_ids + ([tokenizer.pad_token_id] * padding_length)
    attention_mask = attention_mask + ([0] * padding_length)
    input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
    attention_mask = torch.tensor(attention_mask).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)[0].mean(axis=1)
    return output.cpu()

### Generate Embeddings for Job Description and Resumes

In [56]:
# Get the target job description text
target_jd_text = job_descriptions[TARGET_JD_FILENAME]

# Generate embedding for the JD
jd_embedding = get_embedding(target_jd_text, tokenizer, encoder, device)
print('Generated embedding for the target job description.')

Generated embedding for the target job description.


In [57]:
# Generate embeddings for all resumes
resumes_df['embedding'] = resumes_df['Resume_str'].progress_apply(
    lambda x: get_embedding(x, tokenizer, encoder, device)
)
print(f'Generated embeddings for {len(resumes_df)} resumes.')

100%|██████████| 2437/2437 [02:55<00:00, 13.92it/s]

Generated embeddings for 2437 resumes.


### Calculate Similarity and Rank

In [58]:
# Calculate cosine similarity
resumes_df['similarity_score'] = resumes_df['embedding'].progress_apply(
    lambda x: F.cosine_similarity(x, jd_embedding).item()
)

# Sort by similarity score
ranked_resumes = resumes_df.sort_values(by='similarity_score', ascending=False)

100%|██████████| 2437/2437 [00:00<00:00, 57467.53it/s]


In [59]:
# Create better results display with Applicant ID, Similarity Score, and Rank
final_ranking = ranked_resumes[['Person Name', 'similarity_score']].copy()
final_ranking['Rank'] = range(1, len(final_ranking) + 1)

# Rename columns for better display
final_ranking = final_ranking.rename(columns={
    'Person Name': 'Applicant_ID',
    'similarity_score': 'Similarity_Score'
})

# Reorder columns: Rank, Applicant_ID, Similarity_Score
final_ranking = final_ranking[['Rank', 'Applicant_ID', 'Similarity_Score']]

# Round similarity scores to 4 decimal places for better readability
final_ranking['Similarity_Score'] = final_ranking['Similarity_Score'].round(4)

# Save the results
final_ranking.to_csv(RANKING_OUTPUT_PATH, index=False)

print(f'🎯 Ranking complete! Results saved to {RANKING_OUTPUT_PATH}')
print(f'📊 Top 10 candidates ranked by similarity to Web Developer job:')
print("=" * 60)

# Display top 10 with nice formatting
top_10 = final_ranking.head(10)
for _, row in top_10.iterrows():
    print(f"Rank {row['Rank']:2d} | Applicant: {row['Applicant_ID']:15s} | Score: {row['Similarity_Score']:.4f}")

print("=" * 60)
print(f"💾 Full ranking ({len(final_ranking)} candidates) saved to CSV file")

# Return the final ranking for further analysis
final_ranking.head()

🎯 Ranking complete! Results saved to ../data/results/siamese_ranking_results.csv
📊 Top 10 candidates ranked by similarity to Web Developer job:
Rank  1 | Applicant: BQN             | Score: 0.9449
Rank  2 | Applicant: XK              | Score: 0.9422
Rank  3 | Applicant: AFX             | Score: 0.9416
Rank  4 | Applicant: ZO              | Score: 0.9415
Rank  5 | Applicant: XT              | Score: 0.9390
Rank  6 | Applicant: BRC             | Score: 0.9385
Rank  7 | Applicant: BTM             | Score: 0.9368
Rank  8 | Applicant: FI              | Score: 0.9365
Rank  9 | Applicant: BUW             | Score: 0.9363
Rank 10 | Applicant: BTF             | Score: 0.9324
💾 Full ranking (2437 candidates) saved to CSV file


,Rank,Applicant_ID,Similarity_Score
1807,1,BQN,0.9449
634,2,XK,0.9422
855,3,AFX,0.9416
690,4,ZO,0.9415
643,5,XT,0.9390
